# Amaliyot 7. Qarorlar daraxtlari

7-interaktiv laboratoriyaga («Qarorlar daraxtlari») hamroh. Kurs mavzusi: daraxt tuzilishi, boʻlish mezonlari va qoidalar

Daftar oʻz-oʻzidan yetarli: birinchi katak `common.py` faylini oʻquv korpusi va yordamchi funksiyalar bilan yaratadi. Kataklarni yuqoridan pastga ishga tushiring. Korpus oʻquv (shablon) boʻlib, haqiqiy masalada oʻz CSV faylingizni quyidagi ustunlar bilan qoʻying: `text` va `label`.

In [ ]:
%%writefile common.py
"""Matnlarni tasniflash va klasterlash amaliyotlari uchun umumiy funksiyalar."""
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

TOPICS = {
    "sport": ("oʻyin jamoa gol oʻyinchi murabbiy chempionat turnir gʻalaba magʻlubiyat mavsum stadion muxlis hakam hisob "
              "himoyachi hujumchi darvozabon kubok liga final transfer klub zarba penalti musobaqa sportchi "
              "rekord medal olimpiada mashq durang").split(),
    "siyosat": ("qonun deputat saylov partiya ovoz prezident vazir hukumat parlament islohot byudjet "
                "sanksiya nomzod koalitsiya vazirlik farmon tuzatish loyiha muxolifat referendum diplomatiya "
                "hokim konstitutsiya majlis komissiya muzokara sammit saylovchi mandat vakolat").split(),
    "oshxona": ("shoʻrva retsept tuz goʻsht pishirmoq duxovka sabzavot tova qozon xamir shakar ziravor qaynatmoq qovurmoq "
                "dimlamoq sous bulyon salat oshpaz taom masalliq yogʻ un tuxum sut sarimsoq piyoz garmdori "
                "kartoshka shirinlik pishiriq marinad").split(),
    "texnologiya": ("protsessor dastur ilova smartfon neyrotarmoq algoritm maʼlumot server yangilanish dasturchi "
                    "kompyuter xotira ekran kamera brauzer tarmoq internet bulut kod platforma foydalanuvchi qurilma "
                    "chip batareya noutbuk dasturiy videokarta tizim interfeys shifrlash").split(),
}
NEUTRAL = ("bugun juda yangi aytishadi keyin ham mumkin edi boʻldi maʼlum qilindi kecha yaqinda hali hammasi "
           "bir nechta muhim birinchi").split()


def make_corpus(n_per_class=150, noise=0.3, length=(6, 11), seed=0):
    """4 mavzudagi qisqa «hujjatlar»dan iborat oʻquv korpusi.
    noise: oʻz mavzusidan boʻlmagan soʻzlar ulushi (neytral yoki begona). noise qancha katta boʻlsa, masala shuncha qiyin."""
    rng = random.Random(seed)
    names = list(TOPICS)
    rows = []
    for label in names:
        others = [t for t in names if t != label]
        for _ in range(n_per_class):
            words = []
            for _ in range(rng.randint(*length)):
                u = rng.random()
                if u < noise * 0.5:
                    words.append(rng.choice(NEUTRAL))
                elif u < noise:
                    words.append(rng.choice(TOPICS[rng.choice(others)]))
                else:
                    words.append(rng.choice(TOPICS[label]))
            rows.append((" ".join(words), label))
    df = pd.DataFrame(rows, columns=["text", "label"])
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


def project_2d(X, seed=0):
    """TF-IDF siyrak matritsasini tekislikka proyeksiyalash (TruncatedSVD)."""
    from sklearn.decomposition import TruncatedSVD
    return TruncatedSVD(2, random_state=seed).fit_transform(X)


def scatter2d(Z, labels, title="", ax=None):
    ax = ax or plt.gca()
    codes, uniq = pd.factorize(pd.Series(labels))
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=codes, cmap="tab10", s=14, alpha=0.8)
    ax.set_title(title)
    handles = [plt.Line2D([], [], marker="o", ls="", color=sc.cmap(sc.norm(i)), label=str(u)) for i, u in enumerate(uniq)]
    ax.legend(handles=handles, fontsize=8)
    return ax


def top_terms(coef, terms, class_names, n=8):
    """Har bir sinf uchun eng katta vaznli soʻzlarni chiqaradi (chiziqli modellar)."""
    terms = np.asarray(terms)
    for i, name in enumerate(class_names):
        idx = np.argsort(coef[i])[::-1][:n]
        print(f"{name:12s}", ", ".join(terms[idx]))


In [ ]:
from common import *

# Oʻquv korpusi: 4 ta mavzu. noise — "begona" soʻzlar ulushi (qancha koʻp boʻlsa, shuncha qiyin).
df = make_corpus(n_per_class=150, noise=0.45, seed=0)
# Oʻquv korpusi oʻrniga oʻzingiznikini qoʻying: df = pd.read_csv("texts.csv")  # ustunlar: text, label
print(df.label.value_counts().to_dict())
df.head()

## 1. Daraxt qoidalar toʻplami sifatida

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.model_selection import train_test_split

Xtr_t, Xte_t, ytr, yte = train_test_split(df.text, df.label, test_size=0.3, stratify=df.label, random_state=0)
vec = TfidfVectorizer(max_features=300)
Xtr = vec.fit_transform(Xtr_t); Xte = vec.transform(Xte_t)
tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xtr, ytr)
print(export_text(tree, feature_names=list(vec.get_feature_names_out())))
plt.figure(figsize=(14, 6))
plot_tree(tree, feature_names=vec.get_feature_names_out(), class_names=tree.classes_, filled=True, fontsize=8)
plt.show()

## 2. Boʻlish mezonlari: Gini va entropiya

In [ ]:
for crit in ["gini", "entropy"]:
    t = DecisionTreeClassifier(criterion=crit, max_depth=6, random_state=0).fit(Xtr, ytr)
    print(crit, "train:", round(t.score(Xtr, ytr), 3), "test:", round(t.score(Xte, yte), 3))

## 3. Haddan tashqari moslashish: daraxt chuqurligi

In [ ]:
depths = range(1, 21)
tr, te = [], []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(Xtr, ytr)
    tr.append(t.score(Xtr, ytr)); te.append(t.score(Xte, yte))
plt.plot(depths, tr, label="oʻqitish"); plt.plot(depths, te, label="test")
plt.xlabel("max_depth"); plt.ylabel("accuracy"); plt.legend(); plt.title("Chuqurlik va haddan tashqari moslashish"); plt.show()

## 4. Soʻzlar muhimligi

In [ ]:
t = DecisionTreeClassifier(max_depth=6, random_state=0).fit(Xtr, ytr)
imp = pd.Series(t.feature_importances_, index=vec.get_feature_names_out()).sort_values(ascending=False).head(12)
imp[::-1].plot.barh(); plt.title("Daraxtda soʻzlar muhimligi"); plt.show()

## Topshiriqlar

1. Tanlang `min_samples_leaf` va `max_depth`, bunda testdagi toʻgʻrilik maksimal boʻladi.
2. Sinab koʻring `ccp_alpha` (shoxlarni kesish, cost-complexity pruning). Daraxt oʻlchami qanday oʻzgaradi?
3. Eng yuqori tozalikdagi barg uchun qoidani soʻzlar bilan yozing.